In [1]:
import pickle as pkl
import numpy as np
from matplotlib import pyplot as plt
from glob import glob
import pandas as pd
from scipy import stats

In [4]:
import statsmodels.formula.api as smf

In [2]:
df_all = pd.read_parquet("../data/generated/engagement_zero_contrast.pqt")

In [5]:
df_all["is_correct"] = (df_all["feedbackType"] == 1).astype(int)

,engagement,rt,probabilityLeft,feedbackType,signcont,choice,trial_within_block,eid,subject,is_correct
93,0.022990,0.698741,0.8,1.0,0.0,1.0,4,6713a4a7-faed-4df2-acab-ee4e63326f8d,NYU-11,1
122,0.268967,0.336571,0.8,1.0,0.0,1.0,33,6713a4a7-faed-4df2-acab-ee4e63326f8d,NYU-11,1
126,0.224537,0.299414,0.2,1.0,0.0,-1.0,4,6713a4a7-faed-4df2-acab-ee4e63326f8d,NYU-11,1
129,0.138437,1.263167,0.2,-1.0,0.0,1.0,7,6713a4a7-faed-4df2-acab-ee4e63326f8d,NYU-11,0
141,0.195592,0.584131,0.2,1.0,0.0,-1.0,19,6713a4a7-faed-4df2-acab-ee4e63326f8d,NYU-11,1
...,...,...,...,...,...,...,...,...,...,...
1164,0.244646,0.184130,0.8,1.0,0.0,1.0,7,c7bd79c9-c47e-4ea5-aea3-74dda991b48e,CSH_ZAD_029,1
1167,0.104234,0.156384,0.8,1.0,0.0,1.0,10,c7bd79c9-c47e-4ea5-aea3-74dda991b48e,CSH_ZAD_029,1
1177,0.283170,1.779891,0.8,1.0,0.0,1.0,20,c7bd79c9-c47e-4ea5-aea3-74dda991b48e,CSH_ZAD_029,1
1206,0.246810,0.492154,0.2,-1.0,0.0,1.0,18,c7bd79c9-c47e-4ea5-aea3-74dda991b48e,CSH_ZAD_029,0


In [7]:
lmm_model_eid = smf.mixedlm(
    "is_correct ~ engagement + rt", data=df_all, groups=df_all["eid"], re_formula="~1"
)

In [8]:
lmm_results_eid = lmm_model_eid.fit()
print(lmm_results_eid.summary())

          Mixed Linear Model Regression Results
Model:            MixedLM Dependent Variable: is_correct 
No. Observations: 17858   Method:             REML       
No. Groups:       457     Scale:              0.2358     
Min. group size:  7       Log-Likelihood:     -12520.0561
Max. group size:  166     Converged:          Yes        
Mean group size:  39.1                                   
---------------------------------------------------------
               Coef.  Std.Err.   z    P>|z| [0.025 0.975]
---------------------------------------------------------
Intercept       0.597    0.006 94.872 0.000  0.584  0.609
engagement      0.122    0.019  6.550 0.000  0.085  0.158
rt             -0.024    0.008 -2.866 0.004 -0.041 -0.008
Group Var       0.002    0.001                           



/Users/dkundu/mamba/envs/info-decom/lib/python3.10/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


In [9]:
session_data = (
    df_all.groupby("eid")
    .agg(mean_accuracy=("is_correct", "mean"), mean_engagement=("engagement", "mean"))
    .reset_index()
)

In [11]:
import seaborn as sns

In [15]:
import tempfile
import pickle
import sys
import traceback
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from pprint import pprint
from brainbox.io.one import SessionLoader
from prior_localization.functions.utils import compute_mask
from one.api import ONE
from scipy.special import logit, softmax
import os
from os.path import join
import pickle as pkl
from brainwidemap.bwm_loading import bwm_query, bwm_units, load_trials_and_mask, merge_probes


# config = check_config()
one = ONE(
    base_url="https://openalyx.internationalbrainlab.org",
    password="international",
    silent=True,
    username="intbrainlab",
)

bwm_df = bwm_query(one)
runonalleids = bwm_df["eid"].unique()

with open(f"../data/generated//all_eids_engagement.pkl", "rb") as f:
    engagement_pickle = pkl.load(f)

Loading bwm_query results from fixtures/2023_12_bwm_release.csv


In [16]:
limits = []
for eid in engagement_pickle.keys():
    limits.append([np.max(engagement_pickle[eid]), np.min(engagement_pickle[eid])])

In [24]:
limits = np.asarray(limits)